# Day 1: Data Quality KPIs

This notebook measures data quality using practical KPIs.

Learning objectives:
- calculate key quality KPIs,
- compare field-based vs record-based views,
- interpret KPI results for stakeholders.


In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/raw/week2_customer_transactions_messy.csv", "data/raw/week2_customer_transactions_messy.csv"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: data/raw/week2_customer_transactions_messy.csv
Bootstrap complete.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# The file is downloaded to 'data/raw/week2_customer_transactions_messy.csv' relative to the current directory.
# We can load it directly using this relative path.
df = pd.read_csv('data/raw/week2_customer_transactions_messy.csv')
print('Shape:', df.shape)
df.head()

Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09


## KPI formulas

- **Error Rate** = records with >=1 violation / total records
- **Completeness Rate** = non-missing cells / total cells
- **Duplication Rate** = duplicate records / total records
- **Timeliness Rate** = records updated within threshold / records with valid dates
- Optional: weighted overall score


In [3]:
n=len(df)
trx = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
upd = pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')
amount = pd.to_numeric(df['amount'], errors='coerce')
flags = pd.DataFrame({
'missing_customer': df['customer_id'].isna() | (df['customer_id'].astype(str).str.strip()==''),
'invalid_amount': amount.isna() | (amount<0),
'invalid_currency': ~df['currency'].isin(['EUR','USD']),
'missing_payment': df['payment_method'].isna() | (df['payment_method'].astype(str).str.strip()==''),
'invalid_date': trx.isna(),
})
flags['record_error'] = flags.any(axis=1)
flags.head()


,missing_customer,invalid_amount,invalid_currency,missing_payment,invalid_date,record_error
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,True,False,False,False,True
3,True,False,False,False,False,True
4,False,False,True,False,False,True


## Step 1 - Error Rate (record-based)


In [9]:
error_rate = flags['record_error'].mean()
print(f'Error Rate: {error_rate:.2%}')
df[flags['record_error']].head(8)


Error Rate: 54.55%


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN
10,T0010,C109,2026-01-11,52.10,EUR,NaN,completed,NL,2026-01-11


## Step 2 - Completeness Rate (field-based)


In [10]:
total_cells = df.shape[0]*df.shape[1]
missing_cells = df.isna().sum().sum()
completeness = 1-(missing_cells/total_cells)
print(f'Completeness Rate: {completeness:.2%}')


Completeness Rate: 95.96%


## Step 3 - Duplication Rate


In [11]:
dup_full = df.duplicated().mean()
dup_key = df.duplicated(subset=['transaction_id']).mean()
print(f'Full-row duplication: {dup_full:.2%}')
print(f'transaction_id duplication: {dup_key:.2%}')


Full-row duplication: 9.09%
transaction_id duplication: 9.09%


## Step 4 - Timeliness Rate (<= 7 days)


In [12]:
valid = trx.notna() & upd.notna()
lag = (upd-trx).dt.days
timely = valid & (lag>=0) & (lag<=7)
timeliness = timely.sum()/valid.sum()
print(f'Timeliness Rate: {timeliness:.2%}')


Timeliness Rate: 55.56%


## Step 5 - Optional overall quality score


In [13]:
overall = 0.30*completeness + 0.30*(1-error_rate) + 0.20*(1-dup_key) + 0.20*timeliness
print(f'Overall Data Quality Score: {overall:.2%}')


Overall Data Quality Score: 71.72%


## KPI summary table


In [14]:
kpi = pd.DataFrame([
('Error Rate', error_rate, 'Lower is better'),
('Completeness Rate', completeness, 'Higher is better'),
('Duplication Rate (transaction_id)', dup_key, 'Lower is better'),
('Timeliness Rate (<=7 days)', timeliness, 'Higher is better'),
('Overall Score (optional)', overall, 'Higher is better')
], columns=['KPI','Value','Direction'])
kpi['Value (%)']=(kpi['Value']*100).round(2)
kpi[['KPI','Value (%)','Direction']]


,KPI,Value (%),Direction
0,Error Rate,54.55,Lower is better
1,Completeness Rate,95.96,Higher is better
2,Duplication Rate (transaction_id),9.09,Lower is better
3,Timeliness Rate (<=7 days),55.56,Higher is better
4,Overall Score (optional),71.72,Higher is better


## Function spotlight - KPI functions with parameters

Below, we convert KPI formulas into reusable functions.
Pay attention to parameters: they let you adjust thresholds and business rules without rewriting code.


In [15]:
def calculate_completeness_rate(dataframe):
    """Compute field-based completeness rate.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Input table.

    Returns
    -------
    float
        Completeness rate in [0, 1].
    """
    total_cells = dataframe.shape[0] * dataframe.shape[1]
    missing_cells = dataframe.isna().sum().sum()
    return 1 - (missing_cells / total_cells)


def calculate_error_rate(flag_series):
    """Compute record-based error rate from a boolean flag.

    Parameters
    ----------
    flag_series : pd.Series (bool)
        True indicates record has at least one error.

    Returns
    -------
    float
        Error rate in [0, 1].
    """
    return float(flag_series.mean())


def calculate_timeliness_rate(transaction_dates, update_dates, max_delay_days=7):
    """Compute timeliness rate with a configurable delay threshold.

    Parameters
    ----------
    transaction_dates : pd.Series (datetime)
        Parsed business event dates.
    update_dates : pd.Series (datetime)
        Parsed last-updated dates.
    max_delay_days : int, default=7
        Maximum acceptable delay between transaction and update.

    Returns
    -------
    float
        Timeliness rate among records with valid dates.
    """
    valid = transaction_dates.notna() & update_dates.notna()
    lag_days = (update_dates - transaction_dates).dt.days
    timely = valid & (lag_days >= 0) & (lag_days <= max_delay_days)
    return float(timely.sum() / valid.sum())


print('Completeness (function):', round(calculate_completeness_rate(df), 4))
print('Error rate (function):', round(calculate_error_rate(flags['record_error']), 4))
print('Timeliness 7 days (function):', round(calculate_timeliness_rate(trx, upd, max_delay_days=7), 4))
print('Timeliness 3 days (function):', round(calculate_timeliness_rate(trx, upd, max_delay_days=3), 4))


Completeness (function): 0.9596
Error rate (function): 0.5455
Timeliness 7 days (function): 0.5556
Timeliness 3 days (function): 0.5556


## Mini exercises

1. Recalculate timeliness with a 3-day threshold.
2. Which KPI changes most if missing dates increase?
3. For finance reporting, which KPI should have highest weight and why?


## Recap

KPIs help translate data quality into measurable performance signals that support audit and governance decisions.
